In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import numpy as np
from tqdm import tqdm
from xlstm.xlstm_block_stack import xLSTMBlockStack, xLSTMBlockStackConfig
from dacite import from_dict, Config as DaciteConfig
from omegaconf import OmegaConf
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

from math import gcd
from functools import reduce
import torch.nn as nn
import numpy as np
from pandas import read_csv
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import train_test_split
import pandas as pd
import random

In [2]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)
torch.tensor([1.0, 2.0]).cuda()

2.7.1+cu128
True
NVIDIA GeForce RTX 5070 Ti
12.8


tensor([1., 2.], device='cuda:0')

In [ ]:
def ReadExcel(excelpath: str):
    DF = pd.read_excel(excelpath)
    Values = DF.iloc[:, 1:].values 
    
    Values = Values.astype('float32') 
    print(Values.shape)
    return Values

In [ ]:
Vel_1_23_1200 = ReadExcel(r'D:\0DATA\NHRI\1、23_1200.xlsx')
Vel_2_32_300 = ReadExcel(r'D:\0DATA\NHRI\2、32_300.xlsx')
Vel_3_32_160 = ReadExcel(r'D:\0DATA\NHRI\3、32_160.xlsx')
Vel_4_32_sametime = ReadExcel(r'D:\0DATA\NHRI\4、32_sametime.xlsx')
Vel_5_23_160 = ReadExcel(r'D:\0DATA\NHRI\5、23_160.xlsx')
Vel_6_23_300 = ReadExcel(r'D:\0DATA\NHRI\6、23_300.xlsx')
Vel_7_23_600 = ReadExcel(r'D:\0DATA\NHRI\7、23_600.xlsx')
Vel_8_32_600 = ReadExcel(r'D:\0DATA\NHRI\8、32_600.xlsx')
Vel_9_23_900 = ReadExcel(r'D:\0DATA\NHRI\9、23_900.xlsx')
Vel_10_32_900 = ReadExcel(r'D:\0DATA\NHRI\10、32_900.xlsx')
Vel_11_32_1200 = ReadExcel(r'D:\0DATA\NHRI\11、32_1200.xlsx')
Vel_12_23_1500 = ReadExcel(r'D:\0DATA\NHRI\12、23_1500.xlsx')
Vel_13_32_1500 = ReadExcel(r'D:\0DATA\NHRI\13、32_1500.xlsx')
Vel_14_23_450 = ReadExcel(r'D:\0DATA\NHRI\14、23_450.xlsx')
Vel_15_32_450 = ReadExcel(r'D:\0DATA\NHRI\15、32_450.xlsx')
Vel_16_23_750 = ReadExcel(r'D:\0DATA\NHRI\16、23_750.xlsx')
Vel_17_32_750 = ReadExcel(r'D:\0DATA\NHRI\17、32_750.xlsx')
Vel_18_23_1050 = ReadExcel(r'D:\0DATA\NHRI\18、23_1050.xlsx')
Vel_19_32_1050 = ReadExcel(r'D:\0DATA\NHRI\19、32_1050.xlsx')
Vel_20_23_1350 = ReadExcel(r'D:\0DATA\NHRI\20、23_1350.xlsx')
Vel_21_32_1350 = ReadExcel(r'D:\0DATA\NHRI\21、32_1350.xlsx')
Vel_22_32_sametime = ReadExcel(r'D:\0DATA\NHRI\22、32_sametime.xlsx')

(1653, 185)
(1057, 185)
(1040, 185)
(762, 185)
(739, 185)
(687, 185)
(1056, 185)
(1392, 185)
(1353, 185)
(1646, 185)
(1988, 185)
(2009, 185)
(2353, 185)
(910, 185)
(1279, 185)
(1216, 185)
(1671, 185)
(1508, 185)
(1853, 185)
(1868, 185)
(1827, 185)
(836, 185)


In [ ]:
nums = [Vel_1_23_1200.shape[0], Vel_2_32_300.shape[0], Vel_3_32_160.shape[0], Vel_4_32_sametime.shape[0], Vel_5_23_160.shape[0], Vel_6_23_300.shape[0],
        Vel_7_23_600.shape[0], Vel_8_32_600.shape[0], Vel_9_23_900.shape[0], Vel_10_32_900.shape[0], Vel_11_32_1200.shape[0], Vel_12_23_1500.shape[0],
        Vel_13_32_1500.shape[0], Vel_14_23_450.shape[0], Vel_15_32_450.shape[0], Vel_16_23_750.shape[0], Vel_17_32_750.shape[0], Vel_18_23_1050.shape[0],
        Vel_19_32_1050.shape[0], Vel_20_23_1350.shape[0], Vel_21_32_1350.shape[0], Vel_22_32_sametime.shape[0]]
RowNum = sum(nums)
print(f"总行数为 ： {RowNum}")

[1653, 1057, 1040, 762, 739, 687, 1056, 1392, 1353, 1646, 1988, 2009, 2353, 910, 1279, 1216, 1671, 1508, 1853, 1868, 1827, 836] 的所有公约数是: [1]
总行数为 ： 30703


In [ ]:

import random

all_conditions = [Vel_1_23_1200, Vel_2_32_300, Vel_3_32_160, Vel_4_32_sametime, Vel_5_23_160, Vel_6_23_300, Vel_7_23_600, Vel_8_32_600, Vel_9_23_900, Vel_10_32_900,
                  Vel_11_32_1200, Vel_12_23_1500, Vel_13_32_1500, Vel_14_23_450,  Vel_15_32_450, Vel_16_23_750, Vel_17_32_750, Vel_18_23_1050, Vel_19_32_1050,
                  Vel_20_23_1350, Vel_21_32_1350, Vel_22_32_sametime]


train_raw = [Vel_1_23_1200, Vel_2_32_300, Vel_3_32_160, Vel_4_32_sametime, Vel_5_23_160, Vel_6_23_300, Vel_8_32_600, Vel_9_23_900, Vel_10_32_900,
             Vel_11_32_1200, Vel_12_23_1500, Vel_14_23_450,  Vel_15_32_450, Vel_16_23_750, Vel_19_32_1050,
             Vel_20_23_1350, Vel_21_32_1350]
test_raw = [Vel_7_23_600, Vel_13_32_1500, Vel_17_32_750, Vel_18_23_1050, Vel_22_32_sametime]    

print("len(test_raw) = ", len(test_raw))
print("len(train_raw) = ", len(train_raw))
print(len(test_raw) + len(train_raw))

len(test_raw) =  5
len(train_raw) =  17
22


In [ ]:
hist_step = 60   
pred_step = 1    
input_channel = 3   
input_data_num = hist_step * input_channel
output_channel = 5  

In [ ]:

def extract_input_matrix(data_list):
    X_all = np.vstack([d[:, :input_data_num] for d in data_list])  # (N_total, 120)
    X_reshaped = X_all.reshape(-1, hist_step, input_channel)               # (N_total, 40, 3)
    return X_reshaped.reshape(-1, input_channel)                    # → (N_total*40, 3)



def extract_output_matrix(data_list):
    return np.vstack([d[:, input_data_num:] for d in data_list])  # (N_total, 54)

X_input_for_scaler = extract_input_matrix(train_raw)
Y_output_for_scaler = extract_output_matrix(train_raw)

print(X_input_for_scaler.shape)
print(Y_output_for_scaler.shape)

(1396740, 3)
(23279, 5)


In [10]:
scaler_input = MinMaxScaler()
scaler_output = MinMaxScaler()

scaler_input.fit(X_input_for_scaler)
scaler_output.fit(Y_output_for_scaler)

,feature_range,"(0, ...)"
,copy,True
,clip,False


In [ ]:


def normalize_condition_data(data_list):
    normalized = []
    for d in data_list:
        x = d[:, :input_data_num].reshape(-1, hist_step, input_channel)
        x = scaler_input.transform(x.reshape(-1, input_channel)).reshape(-1, hist_step, input_channel)
        y = scaler_output.transform(d[:, input_data_num:])
        normalized.append((x, y))
    return normalized

train_data = normalize_condition_data(train_raw)



test_data = []
for d in test_raw:
    x = d[:, :input_data_num].reshape(-1, hist_step, input_channel)
    x = scaler_input.transform(x.reshape(-1, input_channel)).reshape(-1, hist_step, input_channel)
    y = scaler_output.transform(d[:, input_data_num:])
    test_data.append((x, y))

In [12]:
print(len(train_data))
print(len(test_data))
print(type(train_data[0]))
print(type(test_data[0]))

17
5
<class 'tuple'>
<class 'tuple'>


In [ ]:


class ConditionDataset(Dataset):
    def __init__(self, x_array, y_array):
        self.X = torch.tensor(x_array, dtype=torch.float32)  # (N, 40, 3)
        self.Y = torch.tensor(y_array, dtype=torch.float32)  # (N, 54)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


def build_data_loaders(data, batch_size):
    loaders = []
    for x, y in data:
        ds = ConditionDataset(x, y)
        dl = DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)
        loaders.append(dl)
    return loaders


BatchSize= 64

train_loaders = build_data_loaders(train_data, batch_size=BatchSize) 


test_loader = build_data_loaders(test_data, batch_size=BatchSize)   

print(len(train_loaders))
print(len(test_loader))

17
5


In [ ]:
def safe_mape(y_true, y_pred, epsilon=1e-8, threshold=0.01):
    
    
    y_pred = y_pred.to(y_true.device)
    
    
    mask = torch.abs(y_true) > threshold
    
    
    if mask.sum() == 0:
        return torch.tensor(float('nan'), device=y_true.device)
    
    
    ape = torch.abs((y_true[mask] - y_pred[mask]) / (y_true[mask] + epsilon))
    
    return torch.mean(ape) * 100


In [ ]:


import os
os.environ["CUDA_VISIBLE_DEVICES"] = '0'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [17]:
import time, math
import gc
from torchsummary import summary
from torchtsmixer import TSMixer

In [ ]:

# Create the TSMixer model

model = TSMixer(hist_step, pred_step, input_channel, output_channel, dropout_rate = 0.32, ff_dim = 128, num_blocks = 4)

model = model.to(device=device)
print(model)
from torchsummary import summary
summary(model, input_size = (hist_step, input_channel))

# Loss function and optimizer
num_epochs = 10000
epsilon = 0.0001
max_grad_norm = 1.0 
criterion = nn.MSELoss()

optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, eps=epsilon, weight_decay=1e-5) # lr=0.0001
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, threshold=1e-4, cooldown=2, min_lr=1e-6) 

trainloss_list = []
testloss_list = []
trainMAPE_list = []
testMAPE_list = []
start_time = time.time()

save_dir = r'D:\0DATA\NHRI\1、TSMixer\batchsize=64'

# Training loop
loss_hist = np.zeros(num_epochs)
test_loss_hist = np.zeros(num_epochs)
test_loss_threshold = 0.09
best_test_loss = test_loss_threshold


for epoch in range(num_epochs):
    model.train() 
    for loader in train_loaders:
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)
            y_batch = y_batch.to(device)
            
            # Zero the parameter gradients
            optimizer.zero_grad()
             # Forward pass
            outputs = model(x_batch)
            outputs = outputs.reshape(outputs.shape[0], -1)
            #print("outputs.shape = ", outputs.shape)
            #print("y_batch.shape = ", y_batch.shape)
            trainloss = criterion(outputs, y_batch)
            trainloss_list.append(trainloss.item())
            # Backward pass and optimize
            trainloss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)  
            optimizer.step()
            
    with torch.no_grad():  
        for loader in test_loader:
            for x_batch, y_batch in loader:
                x_batch = x_batch.to(device)
                y_batch = y_batch.to(device)
                y_batch = y_batch.reshape(y_batch.shape[0], -1)
                
                outputs_test = model(x_batch)
                outputs_test = outputs_test.reshape(outputs_test.shape[0], -1)
                testloss = criterion(outputs_test, y_batch)
                testloss_list.append(testloss.item())
            
                testmape = safe_mape(outputs_test, y_batch)
                testMAPE_list.append(testmape.item())
    
    
    ave_train_loss = np.mean(trainloss_list)
    ave_test_loss = np.mean(testloss_list)
    ave_test_mape = np.mean(testMAPE_list)
    trainloss_list = []
    testloss_list = []

    scheduler.step(ave_test_loss)  
    
    if(ave_test_loss < best_test_loss):     
        torch.save(model, os.path.join(save_dir, f'epoch.{epoch:05d}-test_loss.{ave_test_loss:.5f}-test_MAPE.{ave_test_mape:.5f}.pth'))
        best_test_loss = ave_test_loss
        print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {ave_train_loss.item():.5f}, Test Loss: {ave_test_loss.item():.5f}, Test MAPE: {ave_test_mape.item():.5f}%', '  -- Model saved')   
    elif((epoch % 100) == 0):
        torch.save(model, os.path.join(save_dir, f'epoch.{epoch:05d}-test_loss.{ave_test_loss:.5f}-test_MAPE.{ave_test_mape:.5f}.pth'))
        print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {ave_train_loss.item():.5f}, Test Loss: {ave_test_loss.item():.5f}, Test MAPE: {ave_test_mape.item():.5f}%', '  -- Model saved')
    else:
        print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {ave_train_loss.item():.5f}, Test Loss: {ave_test_loss.item():.5f}, Test MAPE: {ave_test_mape.item():.5f}%')
       
    loss_hist[epoch] = ave_train_loss.item()
    test_loss_hist[epoch] = ave_test_loss.item()


torch.save(model, os.path.join(save_dir, f'epoch.{epoch:05d}-test_loss.{ave_test_loss:.5f}-test_MAPE.{ave_test_mape:.5f}.pth'))        
print("Training complete **************************")
training_time = time.time() - start_time
print("Training time: {} ************************".format(training_time))
print(" ")

csv_path = r'D:\0DATA\NHRI\1、TSMixer\TSMixer Train process.csv'
np.savetxt(csv_path, np.column_stack((loss_hist, testloss_list)), delimiter=",", header="TrainLoss,TestLoss", comments='')

TSMixer(
  (mixer_layers): Sequential(
    (0): MixerLayer(
      (time_mixing): TimeMixing(
        (norm): TimeBatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (dropout): Dropout(p=0.4, inplace=False)
        (fc1): Linear(in_features=60, out_features=60, bias=True)
      )
      (feature_mixing): FeatureMixing(
        (norm_before): TimeBatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (norm_after): Identity()
        (dropout): Dropout(p=0.4, inplace=False)
        (fc1): Linear(in_features=3, out_features=128, bias=True)
        (fc2): Linear(in_features=128, out_features=3, bias=True)
        (projection): Identity()
      )
    )
    (1): MixerLayer(
      (time_mixing): TimeMixing(
        (norm): TimeBatchNorm2d(180, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (dropout): Dropout(p=0.4, inplace=False)
        (fc1): Linear(in_features=60, out_features=60, bias=True)
     

KeyboardInterrupt: 